# Notebook 5: Wind Stress Trends over the Southern Ocean



**Kinetic Energy Trends and Eddy Saturation in the Southern Ocean**  

Cristina Martí-Solana, Simón Ruiz, Bàrbara Barceló-Llull, Vincent Combes and Ananda Pascual  

Contact: cmarti@imedea.uib-csic.es



---



## Motivation



The **eddy saturation hypothesis** predicts that under increasing wind forcing,

mesoscale eddy kinetic energy (EKE) rises to compensate, while the time-mean

circumpolar transport remains approximately unchanged (Straub 1993; Hallberg &

Gnanadesikan 2006; Meredith & Hogg 2006).



To test this we need to quantify the **wind stress trend** itself.  This

notebook retrieves the CMEMS blended L4 wind product, computes time series of

area-weighted zonal wind stress (τx) and wind stress magnitude (|τ|) over the

ACC region, and fits robust linear trends for comparison with the EKE trends from

Notebook 2.



### Data Product



| Field | Value |

|---|---|

| **Product ID** | `WIND_GLO_PHY_L4_MY_012_006` |

| **Dataset ID** | `cmems_obs-wind_glo_phy_my_l4_P1M` |

| **Resolution** | 0.25° × 0.25°, monthly |

| **Variables** | `eastward_wind_stress` (τx), `northward_wind_stress` (τy) |

| **Period** | 1992–present |



### Analysis outline



1. Retrieve monthly τx, τy over the Southern Ocean (ACC band)

2. Compute area-weighted zonal-mean τx and |τ| time series

3. Fit Theil–Sen trends and Mann–Kendall significance test

4. Map the spatial pattern of the wind stress trend

5. Per-sector analysis (Indian, Pacific, Atlantic)

6. Correlate wind stress with EKE trends (from Notebook 2)


## 1. Setup and Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ── Copernicus Marine Toolbox ──────────────────────────────────────────────
try:
    import copernicusmarine
    _CMEMS_AVAILABLE = True
except ImportError:
    _CMEMS_AVAILABLE = False
    print("WARNING: copernicusmarine not installed — remote access unavailable.")
    print("         Install with:  pip install copernicusmarine")

if _CMEMS_AVAILABLE:
    copernicusmarine.login()

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))

from utils.regions_and_masks import compute_acc_mask_from_ssh

print(f"Repository root: {REPO_ROOT}")
print(f"Copernicus Marine Toolbox available: {_CMEMS_AVAILABLE}")
print("Imports OK.")

## 2. Configuration

In [ ]:
# ── CMEMS Wind product ─────────────────────────────────────────────────────
WIND_DATASET_ID = "cmems_obs-wind_glo_phy_my_l4_P1M"
WIND_VARIABLES  = ["eastward_wind", "northward_wind"]

# ── Region ─────────────────────────────────────────────────────────────────
LONMIN, LONMAX = -180.0, 180.0
LATMIN, LATMAX = -65, -35

# ── Study period (match Notebook 02 / 04) ─────────────────────────────────
YEAR_START = 2000
YEAR_END   = 2022
years      = np.arange(YEAR_START, YEAR_END + 1)

# ── Ocean-basin sectors (Zhang et al. 2021) ───────────────────────────────
ZHANG_SECTORS = {
    "Indian":   (20,  147),
    "Pacific":  (147, 290),
    "Atlantic": (290, 380),
}

# ── ACC SSH-contour mask ───────────────────────────────────────────────────
USE_ACC_MASK  = True
SSH_ACC_SOUTH = -0.6
SSH_ACC_NORTH =  0.2
MEAN_ADT_PATH = os.path.join(REPO_ROOT, 'outputs', 'mean_adt.nc')

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(REPO_ROOT, 'outputs', 'trends')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Wind dataset : {WIND_DATASET_ID}")
print(f"  Region       : lon [{LONMIN}, {LONMAX}], lat [{LATMIN}, {LATMAX}]")
print(f"  Period       : {YEAR_START}–{YEAR_END}")
print(f"  ACC mask     : {'ON' if USE_ACC_MASK else 'OFF'}")

## 3. Helper Functions

In [ ]:
def lon_in_sector(lon, lo, hi):

    """Check if longitude (°E, ±180) falls inside [lo, hi), handling wrap."""

    lon360 = lon % 360

    lo360  = lo % 360

    hi360  = hi % 360

    if lo360 < hi360:

        return (lon360 >= lo360) & (lon360 < hi360)

    else:

        return (lon360 >= lo360) | (lon360 < hi360)





def make_sector_mask(lons_2d, sectors):

    """Return dict {sector_name: bool mask (lat, lon)}."""

    masks = {}

    for name, (lo, hi) in sectors.items():

        masks[name] = lon_in_sector(lons_2d, lo, hi)

    return masks





def theil_sen(x, y):

    """Theil-Sen robust slope.  Returns: slope, intercept, lo_slope, hi_slope."""

    mask = np.isfinite(y) & np.isfinite(x)

    if mask.sum() < 3:

        return np.nan, np.nan, np.nan, np.nan

    res = stats.theilslopes(np.asarray(y[mask], float), np.asarray(x[mask], float), alpha=0.95)

    return res.slope, res.intercept, res.low_slope, res.high_slope





def mann_kendall(y):

    """Non-parametric Mann-Kendall test.  Returns: tau, p_value."""

    y = np.asarray(y, dtype=float)

    mask = np.isfinite(y)

    if mask.sum() < 4:

        return np.nan, np.nan

    ym = y[mask]

    n  = len(ym)

    s  = sum(np.sign(ym[j] - ym[k]) for k in range(n-1) for j in range(k+1, n))

    var_s = n * (n - 1) * (2 * n + 5) / 18.0

    z = (s - np.sign(s)) / np.sqrt(var_s) if s != 0 else 0.0

    p = 2 * stats.norm.sf(np.abs(z))

    tau = s / (n * (n - 1) / 2.0)

    return tau, p





def _mk_sigstars(p):

    if not np.isfinite(p):

        return ''

    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))





def running_mean(series, window=12):

    """12-month centred running mean (consistent with Notebook 03)."""

    return pd.Series(series).rolling(window, center=True, min_periods=window//2).mean().values





print("Helper functions defined (Theil–Sen + Mann–Kendall).")


## 4. Load ACC Mask

In [ ]:
if USE_ACC_MASK:
    ds_adt = xr.open_dataset(MEAN_ADT_PATH)
    # Accept either 'adt' or 'sla' as the SSH variable name
    ssh_var = [v for v in ds_adt.data_vars
               if any(k in v.lower() for k in ('adt', 'ssh', 'sla', 'zos'))][0]
    mean_adt = ds_adt[ssh_var].values.squeeze()
    adt_lat  = ds_adt['latitude'].values  if 'latitude'  in ds_adt.coords else ds_adt['lat'].values
    adt_lon  = ds_adt['longitude'].values if 'longitude' in ds_adt.coords else ds_adt['lon'].values
    ds_adt.close()

    acc_mask = compute_acc_mask_from_ssh(
        mean_adt,
        lat       = adt_lat,
        lon       = adt_lon,
        ssh_south = SSH_ACC_SOUTH,
        ssh_north = SSH_ACC_NORTH,
    )
    print(f"ACC mask loaded — {int(acc_mask.sum())} cells inside ACC band")
    print(f"  lat: {acc_mask.latitude.values.min():.2f} to {acc_mask.latitude.values.max():.2f}")
    print(f"  lon: {acc_mask.longitude.values.min():.2f} to {acc_mask.longitude.values.max():.2f}")
else:
    acc_mask = None
    print("ACC mask: DISABLED")


## 5. Retrieve Monthly Wind Stress from CMEMS

Stream monthly-mean τx, τy over the Southern Ocean for the study period.
The wind product is already on a monthly grid, so we get one value per
month per grid cell.

In [ ]:
assert _CMEMS_AVAILABLE, "copernicusmarine package required"

print(f"Retrieving wind stress from {WIND_DATASET_ID} …")
print(f"  Period: {YEAR_START}-01-01 to {YEAR_END}-12-31")
print(f"  Region: lon [{LONMIN}, {LONMAX}], lat [{LATMIN}, {LATMAX}]")

ds_wind = copernicusmarine.open_dataset(
    dataset_id        = WIND_DATASET_ID,
    minimum_longitude = LONMIN,
    maximum_longitude = LONMAX,
    minimum_latitude  = LATMIN,
    maximum_latitude  = LATMAX,
    start_datetime    = f"{YEAR_START}-01-01",
    end_datetime      = f"{YEAR_END}-12-31",
    variables         = WIND_VARIABLES,
)

print(f"\nDataset loaded:")
print(f"  Time steps : {ds_wind.dims['time']}")
print(f"  Lat points : {ds_wind.dims['latitude']}")
print(f"  Lon points : {ds_wind.dims['longitude']}")
print(ds_wind)

## 6. Compute Wind Stress Fields

Zonal wind stress (τx) and wind stress magnitude |τ| = √(τx² + τy²).

In [ ]:
# Load into memory
ds_wind.load()

tau_x = ds_wind['eastward_wind']     # (time, lat, lon)  N/m²
tau_y = ds_wind['northward_wind']
tau_mag = np.sqrt(tau_x**2 + tau_y**2)
tau_mag.name = 'wind_stress_magnitude'

# Coordinate arrays
lats = ds_wind['latitude'].values
lons = ds_wind['longitude'].values
times = pd.to_datetime(ds_wind['time'].values)

print(f"τx  range: {float(tau_x.min()):.4f}  to {float(tau_x.max()):.4f} N/m²")
print(f"|τ| range: {float(tau_mag.min()):.4f} to {float(tau_mag.max()):.4f} N/m²")
print(f"Time: {times[0].strftime('%Y-%m')} to {times[-1].strftime('%Y-%m')} ({len(times)} months)")

## 7. Area-Weighted Mean Time Series

Weight each grid cell by cos(latitude) to account for meridian convergence.
Optionally restrict to the ACC band.

In [ ]:
# ── Area weights ───────────────────────────────────────────────────────────
cos_lat = np.cos(np.deg2rad(lats))
weights_2d = np.broadcast_to(cos_lat[:, None], (len(lats), len(lons)))

# ── ACC mask (interpolated to wind grid if needed) ─────────────────────────
if acc_mask is not None:
    # Cast to float first (interp doesn't support bool), then threshold back
    acc_on_wind = (
        acc_mask.astype(float)
        .interp(latitude=lats, longitude=lons, method='nearest')
        .values > 0.5
    )
    spatial_mask = acc_on_wind  # True = inside ACC
    print(f"ACC mask interpolated to wind grid: {spatial_mask.sum()} / {spatial_mask.size} cells")
else:
    spatial_mask = np.ones((len(lats), len(lons)), dtype=bool)

# ── Masked area weights ────────────────────────────────────────────────────
w_masked = np.where(spatial_mask, weights_2d, 0.0)
w_sum    = w_masked.sum()

# ── Time series: area-weighted mean τx and |τ| ─────────────────────────────
n_times = len(times)
ts_tau_x   = np.full(n_times, np.nan)
ts_tau_mag = np.full(n_times, np.nan)

for t in range(n_times):
    tx_t = tau_x.values[t]    # (lat, lon)
    tm_t = tau_mag.values[t]
    # mask NaN ocean cells
    valid = np.isfinite(tx_t) & spatial_mask
    w = np.where(valid, weights_2d, 0.0)
    ws = w.sum()
    if ws > 0:
        ts_tau_x[t]   = np.nansum(tx_t * w) / ws
        ts_tau_mag[t] = np.nansum(tm_t * w) / ws

# Build a DataFrame
df_ts = pd.DataFrame({
    'time':    times,
    'year':    times.year,
    'month':   times.month,
    'tau_x':   ts_tau_x,
    'tau_mag': ts_tau_mag,
})

print(f"Time series length: {len(df_ts)} months")
print(f"Mean τx  : {df_ts['tau_x'].mean():.4f} N/m²")
print(f"Mean |τ| : {df_ts['tau_mag'].mean():.4f} N/m²")
df_ts.head()


## 8. Whole-ACC Wind Stress Trends

In [ ]:
# ── Annual means ───────────────────────────────────────────────────────────

df_annual = df_ts.groupby('year')[['tau_x', 'tau_mag']].mean().reset_index()



fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)



for ax, var, ylabel, title, color in zip(

        axes,

        ['tau_x', 'tau_mag'],

        ['τx (N m⁻²)', '|τ| (N m⁻²)'],

        ['Zonal wind stress (τx)', 'Wind stress magnitude (|τ|)'],

        ['#1f77b4', '#d62728']):



    # Monthly dots + 12-mo running mean

    t_frac = df_ts['year'] + (df_ts['month'] - 0.5) / 12.0

    ax.plot(t_frac, df_ts[var], '.', color=color, ms=3, alpha=0.3, label='monthly')

    rm = running_mean(df_ts[var].values)

    ax.plot(t_frac, rm, '-', color=color, linewidth=2, label='12-mo running mean')



    # Annual Theil–Sen trend

    x_yr = df_annual['year'].values.astype(float)

    y_yr = df_annual[var].values.astype(float)

    ts_sl, ts_int, ts_lo, ts_hi = theil_sen(x_yr, y_yr)

    mk_tau, mk_p = mann_kendall(y_yr)

    sig = _mk_sigstars(mk_p)



    ax.plot(x_yr, ts_sl * x_yr + ts_int, 'k--', linewidth=1.8,

            label=(f"TS {ts_sl*10:+.4f} N m⁻²/10yr "

                   f"(CI [{ts_lo*10:+.4f}, {ts_hi*10:+.4f}])  "

                   f"MK p={mk_p:.3f}{sig}"))



    ax.set_ylabel(ylabel, fontsize=11)

    ax.set_title(title, fontsize=12)

    ax.legend(fontsize=9)

    ax.grid(True, alpha=0.3)



    print(f"\n{title}  ({YEAR_START}–{YEAR_END}):")

    print(f"  Theil–Sen slope : {ts_sl*10:+.5f} N m⁻²/10yr  [{ts_lo*10:+.5f}, {ts_hi*10:+.5f}]")

    print(f"  Mann–Kendall    : tau = {mk_tau:+.3f}, p = {mk_p:.4f} {sig}")



axes[-1].set_xlabel('Year')

plt.suptitle(f'Southern Ocean wind stress  {YEAR_START}–{YEAR_END} (ACC band)',

             fontsize=14, fontweight='bold')

plt.tight_layout()

plt.show()


## 9. Spatial Map of the Wind Stress Trend



At each grid cell, estimate the trend of the annual-mean wind stress with a **Theil–Sen** slope.

For significance, we report **Mann–Kendall** p-values (stipple: $p<0.05$).


In [ ]:
# ── Annual-mean fields per year ────────────────────────────────────────────

# Compute annual means manually to avoid the flox backend (incompatible with Python 3.9)

annual_tau_x_list   = []

annual_tau_mag_list = []



for yr in years:

    yr_mask = pd.to_datetime(ds_wind['time'].values).year == yr

    annual_tau_x_list.append(tau_x.values[yr_mask].mean(axis=0))

    annual_tau_mag_list.append(tau_mag.values[yr_mask].mean(axis=0))



annual_tau_x_arr   = np.array(annual_tau_x_list)    # (n_years, lat, lon)

annual_tau_mag_arr = np.array(annual_tau_mag_list)



n_yr = len(years)

slope_tau_x   = np.full((len(lats), len(lons)), np.nan)

pval_tau_x    = np.full((len(lats), len(lons)), np.nan)

slope_tau_mag = np.full((len(lats), len(lons)), np.nan)

pval_tau_mag  = np.full((len(lats), len(lons)), np.nan)



x_yr = years.astype(float)



print("Computing per-grid-cell Theil–Sen slopes + Mann–Kendall p-values …")

for j in range(len(lats)):

    for i in range(len(lons)):

        y_tx  = annual_tau_x_arr[:, j, i]

        y_tm  = annual_tau_mag_arr[:, j, i]



        if np.isfinite(y_tx).sum() >= 4:

            slope_tau_x[j, i] = theil_sen(x_yr, y_tx)[0]

            pval_tau_x[j, i]  = mann_kendall(y_tx)[1]



        if np.isfinite(y_tm).sum() >= 4:

            slope_tau_mag[j, i] = theil_sen(x_yr, y_tm)[0]

            pval_tau_mag[j, i]  = mann_kendall(y_tm)[1]



print("Done.")



# ── Plot ────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 10),

                          subplot_kw={'projection': ccrs.SouthPolarStereo()})



LONS, LATS = np.meshgrid(lons, lats)



for ax, slope, pval, title, vabs in zip(

        axes,

        [slope_tau_x, slope_tau_mag],

        [pval_tau_x, pval_tau_mag],

        ['Zonal wind stress  τx', 'Wind stress magnitude  |τ|'],

        [0.003, 0.003]):



    ax.set_extent([-180, 180, -80, -30], crs=ccrs.PlateCarree())

    ax.coastlines(linewidth=0.4, color='0.3')

    ax.gridlines(linewidth=0.3, linestyle='--', color='0.6')



    pcm = ax.pcolormesh(LONS, LATS, slope * 10,   # N m⁻² / 10yr

                         transform=ccrs.PlateCarree(),

                         cmap='RdBu_r', vmin=-vabs*10, vmax=vabs*10)



    # Stipple significant cells (MK p < 0.05)

    sig_j, sig_i = np.where(pval < 0.05)

    if len(sig_j):

        ax.scatter(lons[sig_i], lats[sig_j], s=0.3, c='k', alpha=0.3,

                   transform=ccrs.PlateCarree(), rasterized=True)



    # ACC boundary

    if acc_mask is not None:

        ax.contour(acc_mask.longitude.values, acc_mask.latitude.values,

                   acc_mask.values.astype(float), levels=[0.5],

                   colors='k', linewidths=1.0,

                   transform=ccrs.PlateCarree())



    cb = plt.colorbar(pcm, ax=ax, orientation='vertical',

                      pad=0.04, fraction=0.025, shrink=0.8)

    cb.set_label('Trend (N m⁻² / 10yr)', fontsize=9)

    ax.set_title(f'{title} — Theil–Sen trend  (dots: MK p < 0.05)', fontsize=11)



plt.suptitle(f'Spatial wind stress trend  {YEAR_START}–{YEAR_END}',

             fontsize=13, fontweight='bold')

plt.tight_layout()

plt.show()


## 10. Per-Sector Wind Stress Trends

In [ ]:
LONS_2D, LATS_2D = np.meshgrid(lons, lats)

sector_masks = make_sector_mask(LONS_2D, ZHANG_SECTORS)

sector_names = list(ZHANG_SECTORS.keys())

sector_colors = {'Indian': '#2ca02c', 'Pacific': '#1f77b4', 'Atlantic': '#d62728'}



sector_ts = {}



for sname in sector_names:

    smask = sector_masks[sname]

    if acc_mask is not None:

        smask = smask & acc_on_wind



    w = np.where(smask, weights_2d, 0.0)

    ws = w.sum()



    tx_ts  = np.full(n_times, np.nan)

    tm_ts  = np.full(n_times, np.nan)

    for t in range(n_times):

        valid = np.isfinite(tau_x.values[t]) & smask

        wv = np.where(valid, weights_2d, 0.0)

        wsv = wv.sum()

        if wsv > 0:

            tx_ts[t] = np.nansum(tau_x.values[t] * wv) / wsv

            tm_ts[t] = np.nansum(tau_mag.values[t] * wv) / wsv



    sector_ts[sname] = {'tau_x': tx_ts, 'tau_mag': tm_ts}



# ── Plot per-sector ────────────────────────────────────────────────────────

fig, axes = plt.subplots(len(sector_names), 2, figsize=(16, 4*len(sector_names)),

                          sharex=True)



t_frac = df_ts['year'].values + (df_ts['month'].values - 0.5) / 12.0



print(f"{'Sector':<12} {'Variable':<8} {'TS slope/10yr':>14} {'CI95%':>24} {'MK tau':>8} {'MK p':>8}")

print('─' * 80)



for row, sname in enumerate(sector_names):

    for col, (var, ylabel, ckey) in enumerate([

            ('tau_x', 'τx (N m⁻²)', 'tau_x'),

            ('tau_mag', '|τ| (N m⁻²)', 'tau_mag')]):



        ax = axes[row, col]

        ts_data = sector_ts[sname][ckey]

        c = sector_colors[sname]



        ax.plot(t_frac, ts_data, '.', color=c, ms=3, alpha=0.3)

        rm = running_mean(ts_data)

        ax.plot(t_frac, rm, '-', color=c, linewidth=2)



        # Annual Theil–Sen trend

        ann = pd.DataFrame({'year': df_ts['year'], 'val': ts_data}).groupby('year')['val'].mean()

        x = ann.index.values.astype(float)

        y = ann.values

        sl, intc, lo, hi = theil_sen(x, y)

        trend_line = sl * x + intc

        ax.plot(ann.index, trend_line, 'k--', linewidth=1.5)



        tau_mk, mk_p = mann_kendall(y)

        sig = _mk_sigstars(mk_p)



        ax.set_ylabel(ylabel, fontsize=9)

        ax.set_title(f'{sname} — {ylabel}', fontsize=10)

        ax.grid(True, alpha=0.3)



        print(f"{sname:<12} {var:<8} {sl*10:>+14.5f} [{lo*10:+.5f}, {hi*10:+.5f}] {tau_mk:>+8.3f} {mk_p:>8.4f} {sig}")



axes[-1, 0].set_xlabel('Year')

axes[-1, 1].set_xlabel('Year')

plt.suptitle(f'Per-sector wind stress  {YEAR_START}–{YEAR_END}',

             fontsize=13, fontweight='bold')

plt.tight_layout()

plt.show()


## 10b. Combined Overview: Whole SO and Sectors

In [ ]:
eke_path = os.path.join(REPO_ROOT, 'outputs', 'trends', 'energy_field_timeseries.csv')



if not os.path.exists(eke_path):

    print(f"EKE file not found: {eke_path}\nRun Notebook 02 first.")

else:

    df_eke_all = pd.read_csv(eke_path)

    print(f"EKE sectors in file: {df_eke_all['sector'].unique().tolist()}")



    # ── Use pre-computed whole-SO sector directly ───────────────────────────

    df_eke_global = (df_eke_all[df_eke_all['sector'] == 'Whole SO']

                     [['year', 'month', 'mean_eke']]

                     .rename(columns={'mean_eke': 'eke'})

                     .reset_index(drop=True))



    # ── Per-sector EKE — each sector filtered individually ──────────────────

    eke_by_sector = {}

    for sname in sector_names:   # ['Indian', 'Pacific', 'Atlantic']

        sub = (df_eke_all[df_eke_all['sector'] == sname]

               [['year', 'month', 'mean_eke']]

               .rename(columns={'mean_eke': 'eke'})

               .reset_index(drop=True))

        eke_by_sector[sname] = sub

        print(f"  {sname}: {len(sub)} rows")



    # ── Build panel data: [Whole ACC, Indian, Pacific, Atlantic] ────────────

    panel_labels = ['Whole ACC'] + sector_names

    wind_series  = [df_ts[['year', 'month', 'tau_x']].rename(columns={'tau_x': 'tau'})]

    eke_series   = [df_eke_global]

    panel_colors = {'Whole ACC': '#333333',

                    'Indian':    '#2ca02c',

                    'Pacific':   '#1f77b4',

                    'Atlantic':  '#d62728'}



    for sname in sector_names:

        wind_series.append(

            pd.DataFrame({'year':  df_ts['year'].values,

                          'month': df_ts['month'].values,

                          'tau':   sector_ts[sname]['tau_x']})

        )

        eke_series.append(eke_by_sector[sname])



    # ── 2×2 panel figure ────────────────────────────────────────────────────

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

    axes_flat  = list(axes.flat)



    for ax, label, wdf, edf in zip(axes_flat, panel_labels, wind_series, eke_series):

        c = panel_colors[label]



        merged = wdf.merge(edf, on=['year', 'month'], how='inner')

        if merged.empty:

            ax.set_title(f'{label} — no overlapping data', fontsize=11)

            continue



        t_f = merged['year'] + (merged['month'] - 0.5) / 12.0



        # τx (left axis)

        ax.set_ylabel('τx  (N m⁻²)', fontsize=9, color=c)

        ax.tick_params(axis='y', labelcolor=c)

        ax.plot(t_f, merged['tau'], '.', color=c, ms=3, alpha=0.2)

        rm_w = running_mean(merged['tau'].values)

        ax.plot(t_f, rm_w, '-', color=c, linewidth=2, label='τx 12-mo RM')



        ann_w = merged.groupby('year')['tau'].mean()

        xw = ann_w.index.values.astype(float)

        yw = ann_w.values

        sl_w, intc_w, lo_w, hi_w = theil_sen(xw, yw)

        mk_tw, mk_pw = mann_kendall(yw)

        sig_w = _mk_sigstars(mk_pw)

        ax.plot(ann_w.index, sl_w * xw + intc_w,

                '--', color=c, linewidth=1.8,

                label=f"τx {sl_w*10:+.4f}/10yr (CI [{lo_w*10:+.4f},{hi_w*10:+.4f}])  MK p={mk_pw:.3f}{sig_w}")



        # EKE (right axis)

        ax2 = ax.twinx()

        eke_c = '#ff7f0e'

        ax2.set_ylabel('EKE  (m² s⁻²)', fontsize=9, color=eke_c)

        ax2.tick_params(axis='y', labelcolor=eke_c)

        ax2.plot(t_f, merged['eke'], '.', color=eke_c, ms=3, alpha=0.2)

        rm_e = running_mean(merged['eke'].values)

        ax2.plot(t_f, rm_e, '-', color=eke_c, linewidth=2, label='EKE 12-mo RM')



        ann_e = merged.groupby('year')['eke'].mean()

        xe = ann_e.index.values.astype(float)

        ye = ann_e.values

        sl_e, intc_e, lo_e, hi_e = theil_sen(xe, ye)

        mk_te, mk_pe = mann_kendall(ye)

        sig_e = _mk_sigstars(mk_pe)

        ax2.plot(ann_e.index, sl_e * xe + intc_e,

                 '--', color=eke_c, linewidth=1.8,

                 label=f"EKE {sl_e*10:+.5f}/10yr (CI [{lo_e*10:+.5f},{hi_e*10:+.5f}])  MK p={mk_pe:.3f}{sig_e}")



        # Combined legend

        lines1, labs1 = ax.get_legend_handles_labels()

        lines2, labs2 = ax2.get_legend_handles_labels()

        ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc='upper left')



        ax.set_title(label, fontsize=12, fontweight='bold', color=c)

        ax.grid(True, alpha=0.25)



        print(f"{label:<12}  τx {sl_w*10:+.4f} N/m²/10yr (MK p={mk_pw:.4f}){sig_w}"

              f"  |  EKE {sl_e*10:+.5f} m²/s²/10yr (MK p={mk_pe:.4f}){sig_e}"

              f"  ({len(merged)} months)")



    for ax in axes[-1, :]:

        ax.set_xlabel('Year', fontsize=10)



    plt.suptitle(f'Wind stress (τx) and EKE — Whole ACC and per sector  {YEAR_START}–{YEAR_END}',

                 fontsize=13, fontweight='bold')

    plt.tight_layout()

    plt.show()


## 11. Zonal-Mean Wind Stress Trend Profile



Latitude profile of the τx **Theil–Sen** slope — does the westerly jet shift poleward or intensify in place?


In [ ]:
# ── Zonal-mean τx per year ─────────────────────────────────────────────────

zonal_annual_tx = annual_tau_x_arr.mean(axis=2)   # (n_years, lat)



zonal_slope = np.full(len(lats), np.nan)

zonal_pval  = np.full(len(lats), np.nan)



for j in range(len(lats)):

    y_j = zonal_annual_tx[:, j]

    if np.isfinite(y_j).sum() >= 4:

        zonal_slope[j] = theil_sen(x_yr, y_j)[0]

        zonal_pval[j]  = mann_kendall(y_j)[1]



# ── Plot ────────────────────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))



# Left: mean zonal wind stress profile

mean_zonal_tx = tau_x.mean(dim=('time', 'longitude')).values

ax1.plot(mean_zonal_tx, lats, 'k-', linewidth=2)

ax1.axvline(0, color='0.5', linewidth=0.5)

ax1.set_xlabel('Mean τx (N m⁻²)', fontsize=11)

ax1.set_ylabel('Latitude', fontsize=11)

ax1.set_title('Mean zonal wind stress profile', fontsize=12)

ax1.grid(True, alpha=0.3)



# Right: slope profile

colors = np.where(zonal_pval < 0.05, 'red', '0.5')

ax2.barh(lats, zonal_slope * 10, height=0.22, color=colors, edgecolor='none')

ax2.axvline(0, color='k', linewidth=0.5)

ax2.set_xlabel('τx trend  (N m⁻² / 10yr)', fontsize=11)

ax2.set_ylabel('Latitude', fontsize=11)

ax2.set_title('Theil–Sen trend of zonal-mean τx  (red: MK p < 0.05)', fontsize=12)

ax2.grid(True, alpha=0.3)



plt.suptitle(f'Zonal-mean wind stress  {YEAR_START}–{YEAR_END}',

             fontsize=13, fontweight='bold')

plt.tight_layout()

plt.show()


## 13. Save Wind Stress Time Series

In [ ]:
out_dir = os.path.join(REPO_ROOT, 'outputs', 'trends')

os.makedirs(out_dir, exist_ok=True)

out_csv = os.path.join(out_dir, 'wind_stress_trends.csv')



rows = []



def _annual_trend(df, col):

    ann = df.groupby('year')[col].mean()

    x = ann.index.values.astype(float)

    y = ann.values.astype(float)

    slope, intercept, lo, hi = theil_sen(x, y)

    mk_tau, mk_p = mann_kendall(y)

    return slope, lo, hi, mk_tau, mk_p



# Whole ACC trends (area-weighted TS computed earlier)

for col, label in [('tau_x', 'tau_x'), ('tau_mag', 'tau_mag')]:

    slope, lo, hi, mk_tau, mk_p = _annual_trend(df_ts, col)

    rows.append({

        'sector': 'Whole ACC',

        'variable': label,

        'theil_sen_slope_per_year': slope,

        'theil_sen_ci95_low_per_year': lo,

        'theil_sen_ci95_high_per_year': hi,

        'mann_kendall_tau': mk_tau,

        'mann_kendall_p': mk_p,

        'significant_05': bool(np.isfinite(mk_p) and (mk_p < 0.05)),

    })



# Per-sector trends

for sname in sector_names:

    for key, label in [('tau_x', 'tau_x'), ('tau_mag', 'tau_mag')]:

        tmp = pd.DataFrame({'year': df_ts['year'].values,

                            'month': df_ts['month'].values,

                            key: sector_ts[sname][key]})

        slope, lo, hi, mk_tau, mk_p = _annual_trend(tmp, key)

        rows.append({

            'sector': sname,

            'variable': label,

            'theil_sen_slope_per_year': slope,

            'theil_sen_ci95_low_per_year': lo,

            'theil_sen_ci95_high_per_year': hi,

            'mann_kendall_tau': mk_tau,

            'mann_kendall_p': mk_p,

            'significant_05': bool(np.isfinite(mk_p) and (mk_p < 0.05)),

        })



df_trends = pd.DataFrame(rows)

df_trends.to_csv(out_csv, index=False)



print(f"Saved: {out_csv}")

display(df_trends)


## Summary



This notebook computes Southern Ocean wind stress time series and trends.



- Trends are estimated with **Theil–Sen** (robust slope; includes 95% CI on slope).

- Trend significance is assessed with **Mann–Kendall** (non-parametric p-value).

- Outputs written to `outputs/trends/wind_stress_trends.csv`.
